In [ ]:
import time
# Let starter.py compute 12 hours from its own start
global_end_time = 0.0

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    if not (isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)):
        return False
    if np.any((guess < 0) | (guess > 9)):
        return False
    return True
    
def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Invalid permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            # Reject empty or jagged grids
            if not by_rows:
                return None
            row_len = len(by_rows[0])
            if not all(len(r) == row_len for r in by_rows):
                return None
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception as e:
            print(f"[DEBUG] convert_tokens_to_array failed: {type(e).__name__}: {e}")
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))
    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not query['train']:
                    break
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            num_guesses = len(v)
            for i in range(len(target_dict)):
                if i < num_guesses:
                    target_dict[f'attempt_{i+1}'] = v[i].tolist()
                else:
                    target_dict[f'attempt_{i+1}'] = v[-1].tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs


def getter_full_probmul_3(guesses, baseline=None):
    # Dynamic baseline to avoid negative scores (from V1)
    if baseline is None:
        beam_scores = [g["beam_score"] for g in guesses]
        if not beam_scores:
            return 0
        baseline = max(beam_scores) + 1.0
    inf_score = np.sum([baseline - g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline - s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_dual_selection(self):
        """Return {base_key: [best_from_probmul, best_alternative_from_pool]}.

        Default: use top-3 candidates from both scoring algorithms.
        Debug-only: use top-8 for known solved-not-dual puzzles.
        Attempt 1: score_full_probmul_3's top pick.
        Attempt 2: best unique candidate from the selected pool that is not attempt 1.
        """
        results = {}

        # Debug experiment only. Keep this empty before full LB submission.
        debug_deep_pool_keys = {
             "4c7dc4dd",
             "7b0280bc",
             "2d0172a1",
             "31f7f899",
             "3dc255db",
             "8f3a5a89",
             "800d221b",
        }

        for bk, guesses in self.decoded_results.items():
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            ranked_b = score_kgmon(g)

            base_key = bk.split("_")[0]
            pool_k = 8 if base_key in debug_deep_pool_keys else 3

            candidates_pool = []
            seen = set()
            for cand in ranked_a[:pool_k] + ranked_b[:pool_k]:
                h = hashable(cand)
                if h not in seen:
                    seen.add(h)
                    candidates_pool.append(cand)

            top_a = candidates_pool[0] if candidates_pool else None
            top_b = candidates_pool[1] if len(candidates_pool) > 1 else top_a

            combined = [x for x in [top_a, top_b] if x is not None]
            results[bk] = combined

        return results
        
    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.rsplit("_", 1)
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg correct beam score: N/A (no correct solutions found)")
            print(" max correct beam score: N/A (no correct solutions found)")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.rsplit("_", 1)[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

        # Dual selection benchmark (this is what our actual submission uses)
        dual_selected = self.run_dual_selection()
        correct_puzzles_dual = {
            k for k, v in dual_selected.items()
            if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])
        }
        print(correct_puzzles_dual)
        score_dual = sum(1/num_tasks_per_puzzle[k.rsplit("_", 1)[0]] for k in correct_puzzles_dual)
        print(f" acc: {score_dual:5.1f}/{num_puzzles:3} ('run_dual_selection — ACTUAL SUBMISSION METHOD')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import copy
import queue
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

import hashlib

def deterministic_seed(s, mod=10000):
    """
    Python's built-in hash() is randomized by PYTHONHASHSEED,
    so it gives different values across processes/runs.
    This function provides a consistent, reproducible seed across all runs.
    """
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % mod

logging.disable(logging.WARNING)

# ------------------------------------------------------------
# Fixed token IDs (as used in fine-tuning)
# ------------------------------------------------------------
ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_BEAM_CANDIDATES = 5
class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            # Ensure labels is 1D (avoid scalar tensor)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            # Use torch.nonzero instead of torch.where to avoid bool scalar issues
            user_mask = (labels == USER_TOKEN_ID)
            assistant_mask = (labels == ASSISTANT_TOKEN_ID)
            user_start_idx = torch.nonzero(user_mask, as_tuple=True)[0].tolist()
            assistant_start_idx = torch.nonzero(assistant_mask, as_tuple=True)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            eos_mask = (labels == EOS_ID)
            end_idx = torch.nonzero(eos_mask, as_tuple=True)[0].tolist()
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                if start >= end:
                    continue
                if j % 2 == 1:  # assistant response part
                    start += 2
                    end += 1
                    if start < end and end <= labels.size(0):
                        batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        # Direct cache pass (fast, no clone)
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,
            start_time=start_time,
            end_time=end_time,
        )

        del outputs
        # torch.cuda.empty_cache() removed for speed

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    
    max_len = max(batch_lengths)
    padded_tokens = []
    attention_masks = []
    for tokens in batch_tokens:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        padded_tokens.append(padded)
        attention_masks.append([1] * len(tokens) + [0] * pad_len)
        
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(attention_masks, device=model.device, dtype=torch.long)
    
    # FIX: use_cache=False because padding breaks KV cache
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    
    result = []
    # FIX: Slicing on GPU before transferring to CPU to prevent massive OOM Crash
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_logits = outputs.logits[i, query_length-1 : query_length-1+len(answer_tokens), :]
        answer_log_probs = answer_logits.float().log_softmax(-1).cpu()
        answer_score = answer_log_probs[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
        
    return result


def worker(rank, p_queue, end_time):
    # CUDA optimizations (from winning solution)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch_fused",  # Fused optimizer for speed
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,            # Force bfloat16
        attn_implementation="sdpa",      # Use SDPA (fast, memory efficient)
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    # Fixed queue race condition with Timeout
    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        try:
            key = p_queue.get(timeout=60)
        except queue.Empty:
            continue
            
        if key is None:
            break

        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        set_peft_model_state_dict(model, default_weights.copy(), adapter_name="default")
        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        # Puzzle-specific seed for diversity (deterministic across runs/processes)
        train_seed = deterministic_seed(key)
        print(f"[Rank {rank}] train_seed={train_seed} for puzzle {key}")
        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=train_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        torch.cuda.reset_peak_memory_stats()
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        # FIX: Use puzzle-specific, deterministic seed for evaluation augmentation
        eval_seed = deterministic_seed(f"{key}_eval")
        print(f"[Rank {rank}] eval_seed={eval_seed} for puzzle {key}")
        eval_ds = puzzle_ds_multi.augment(n=2, seed=eval_seed)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
   
        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)

        with torch.inference_mode():
            known_scores = {}

            for subkeys in batches:
                spend_time = time.time() - start_time
                if spend_time > 1600 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(
                    model, tokens, max_new_tokens, max_score, end_time
                )

                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:
                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_seed = hash(bk) % 1024**2
                            aug_dataset = aug_dataset.augment(seed=aug_seed)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if decoded_result:
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import traceback
import numpy as np
import torch.multiprocessing as mp


NPROCS = 4

DEBUG_PUZZLES = [
    # no-output / infrastructure or decode-save suspects
    "271d71e2",  # NO OUTPUT, easy_static/rare
    "291dc1e1",  # NO OUTPUT or unstable, hard_tier1/dynamic/rare

    # dynamic / generation bottleneck
    "136b0064",  # UNSOLVED, hard_tier1/dynamic/rare
    "269e22fb",  # UNSOLVED, dynamic + multi-query
    "0934a4d8",  # UNSOLVED, tier3/dynamic/high_ratio/extreme_ratio
    "20a9e565",  # UNSOLVED, tier3/dynamic/multi-query/high_ratio/extreme_ratio
    "89565ca0",  # UNSOLVED, tier3/dynamic/high_ratio/extreme_ratio

    # solved but dual missed -> selection-depth test
    "4c7dc4dd",  # SOLVED, Dual No in 48-run, rank 4; tier3/multi-query/extreme_ratio
    "7b0280bc",  # SOLVED, sometimes Dual No; rank 1-2 depending run
    "2d0172a1",  # SOLVED, Dual No, rank 2/2

    # extra solved but dual missed -> selection-depth test
    "31f7f899",  # SOLVED, Dual No, rank 2/2
    "3dc255db",  # SOLVED, Dual No, rank 3/3
    "8f3a5a89",  # SOLVED, Dual No, rank 2/2
    "800d221b",  # SOLVED, Dual No, rank 2/2

    # solved + dual yes -> guardrail, ensure selection change does not regress
    "6e453dd6",  # SOLVED, Dual Yes
    "7b80bb43",  # SOLVED, Dual Yes
    "1818057f",  # SOLVED, Dual Yes
    "4a21e3da",  # SOLVED, Dual Yes, multi-query

    # unsolved but useful buckets
    "cb2d8a2c",  # UNSOLVED, easy_static + multi-query
    "58490d8a",  # UNSOLVED, dynamic + high_ratio-ish
]

def get_puzzle_priority(key, data):
    train = data[key]["train"]
    n_train = len(train)

    avg_input_size = np.mean([
        np.prod(np.shape(x["input"])) for x in train
    ])

    size_ratios = [
        max(np.prod(np.shape(x["output"])),
            np.prod(np.shape(x["input"]))) /
        max(min(np.prod(np.shape(x["output"])),
                np.prod(np.shape(x["input"]))), 1)
        for x in train
    ]

    max_ratio = max(size_ratios)
    avg_ratio = np.mean(size_ratios)

    n_colors = len(set(
        int(c)
        for x in train
        for arr in [x["input"], x["output"]]
        for c in np.unique(arr).tolist()
    ))

    is_dynamic = any(
        np.shape(x["input"]) != np.shape(x["output"])
        for x in train
    )

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )

    if n_train >= 6 and tier < 3:
        fine *= 0.7

    return (tier, fine)


def local_worker(rank, queue, end_time):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    # Avoid fragile file-based worker chain. A small stagger reduces startup pressure.
    if rank > 0:
        time.sleep(15 * rank)

    try:
        from arc_solver import worker

        print(f"[Rank {rank}] start!", flush=True)
        worker(rank, queue, end_time)
        print(f"[Rank {rank}] done!", flush=True)

    except Exception as e:
        print(
            f"[Rank {rank}] WORKER_EXCEPTION {type(e).__name__}: {e}",
            flush=True,
        )
        traceback.print_exc()
        return


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--end-time",
        type=float,
        default=0.0,
        help="Unix timestamp when execution must stop. If 0, use 12 hours from now.",
    )
    args = parser.parse_args()

    if args.end_time == 0.0:
        end_time = time.time() + 12 * 3600
    else:
        end_time = args.end_time

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    keys_sorted = sorted(
        data.keys(),
        key=lambda k: get_puzzle_priority(k, data),
    )

    tiers = [get_puzzle_priority(k, data)[0] for k in keys_sorted]
    print(
        f"[Starter] Puzzle tiers - "
        f"Easy: {tiers.count(1)}, "
        f"Hard: {tiers.count(2)}, "
        f"Extreme: {tiers.count(3)}",
        flush=True,
    )
    print(
        f"[Starter] First: {keys_sorted[0]}, Last: {keys_sorted[-1]}",
        flush=True,
    )

    if rerun_mode:
        selected_keys = keys_sorted
    else:
        debug_set = set(DEBUG_PUZZLES)
        selected_keys = [k for k in keys_sorted if k in debug_set]
        missing = [k for k in DEBUG_PUZZLES if k not in data]
        print(f"[Starter] Manual debug mode: {len(selected_keys)} puzzles selected", flush=True)
        print(f"[Starter] Selected: {selected_keys}", flush=True)
        if missing:
            print(f"[Starter] Missing DEBUG_PUZZLES: {missing}", flush=True)

    for key in selected_keys:
        queue.put(key)

    for _ in range(NPROCS):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, end_time), nprocs=NPROCS)

In [ ]:
%%writefile arc_dsl_sidecar.py
import numpy as np


def valid_grid(x):
    x = np.asarray(x)
    return x.ndim == 2 and 0 < x.shape[0] <= 30 and 0 < x.shape[1] <= 30 and np.all((0 <= x) & (x <= 9))


def bbox_nonzero(a):
    a = np.asarray(a)
    ys, xs = np.where(a != 0)
    if len(ys) == 0:
        return None
    return a[ys.min():ys.max()+1, xs.min():xs.max()+1]


def bbox_color(a, color):
    a = np.asarray(a)
    ys, xs = np.where(a == color)
    if len(ys) == 0:
        return None
    return a[ys.min():ys.max()+1, xs.min():xs.max()+1]


def color_map_rule(inp, out):
    if inp.shape != out.shape:
        return None
    mp = {}
    for a, b in zip(inp.ravel(), out.ravel()):
        a = int(a)
        b = int(b)
        if a in mp and mp[a] != b:
            return None
        mp[a] = b
    return mp


def apply_color_map(a, mp):
    res = np.array(a, copy=True)
    for k, v in mp.items():
        res[a == k] = v
    return res


def verify_rule(train, fn):
    for ex in train:
        pred = fn(np.asarray(ex["input"], dtype=int))
        target = np.asarray(ex["output"], dtype=int)
        if not valid_grid(pred):
            return False
        if pred.shape != target.shape:
            return False
        if not np.array_equal(pred, target):
            return False
    return True


def propose_dsl_candidates(task):
    train = task["train"]
    test_in = np.asarray(task["test"][0]["input"], dtype=int)
    candidates = []

    rules = []

    rules.append(("identity", lambda x: np.asarray(x, dtype=int)))

    for k in [1, 2, 3]:
        rules.append((f"rot90_{k}", lambda x, k=k: np.rot90(x, k=k)))
    rules.append(("flipud", lambda x: np.flipud(x)))
    rules.append(("fliplr", lambda x: np.fliplr(x)))
    rules.append(("transpose", lambda x: np.swapaxes(x, 0, 1)))

    rules.append(("bbox_nonzero", lambda x: bbox_nonzero(x)))

    for color in range(1, 10):
        rules.append((f"bbox_color_{color}", lambda x, color=color: bbox_color(x, color)))

    # Consistent direct color map for same-size tasks.
    mp = None
    ok = True
    for ex in train:
        inp = np.asarray(ex["input"], dtype=int)
        out = np.asarray(ex["output"], dtype=int)
        cur = color_map_rule(inp, out)
        if cur is None:
            ok = False
            break
        if mp is None:
            mp = dict(cur)
        else:
            for k, v in cur.items():
                if k in mp and mp[k] != v:
                    ok = False
                    break
                mp[k] = v
        if not ok:
            break
    if ok and mp is not None:
        rules.append(("color_map", lambda x, mp=mp: apply_color_map(x, mp)))

    seen = set()
    for name, fn in rules:
        try:
            if verify_rule(train, fn):
                pred = fn(test_in)
                if valid_grid(pred):
                    key = tuple(map(tuple, np.asarray(pred, dtype=int)))
                    if key not in seen:
                        seen.add(key)
                        candidates.append((name, np.asarray(pred, dtype=int)))
        except Exception:
            pass

    return candidates


def add_dsl_sidecar_candidates(decoder, split_data, only_when_missing=True, max_per_task=2):
    added = {}
    for key in split_data.keys:
        if only_when_missing and key in decoder.decoded_results and decoder.decoded_results[key]:
            continue

        task = split_data.queries[key]
        cands = propose_dsl_candidates(task)
        if not cands:
            continue

        decoder.decoded_results.setdefault(key, {})
        added[key] = []

        for i, (name, sol) in enumerate(cands[:max_per_task]):
            decoder.decoded_results[key][f"{key}.dsl.{name}.{i}"] = {
                "beam_score": 0.0,
                "score_aug": [0.0] * 8,
                "solution": sol,
            }
            added[key].append(name)

    print("DSL sidecar added:", added)
    return added

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

split_data = data.split_multi_replies()
decoder = ArcDecoder(split_data, n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

from arc_dsl_sidecar import add_dsl_sidecar_candidates

add_dsl_sidecar_candidates(
    decoder=decoder,
    split_data=split_data,
    only_when_missing=True,
    max_per_task=2,
)

submission = data.get_submission(decoder.run_dual_selection())

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

In [ ]:
if decoder.decoded_results:
    test_key = list(decoder.decoded_results.keys())[0]
    result = decoder.run_dual_selection()
    print(f"Key: {test_key}")
    print(f"Attempt 1 shape: {result[test_key][0].shape if result[test_key] else None}")
    print(f"Attempt 2 shape: {result[test_key][1].shape if len(result[test_key]) > 1 else None}")
else:
    print("No results were generated. Model inference likely failed.")

In [ ]:
# ================================================================
# DEBUG PUZZLES - SELECTION EXPERIMENT DIAGNOSTIC
# ================================================================
from starter import get_puzzle_priority
from arc_decoder import score_full_probmul_3, score_kgmon, hashable
import numpy as np
import re

if hasattr(data, "queries"):
    data_dict = data.queries
else:
    data_dict = data

DEBUG_PUZZLES = [
    # no-output / infrastructure or decode-save suspects
    "271d71e2",  # NO OUTPUT, easy_static/rare
    "291dc1e1",  # NO OUTPUT or unstable, hard_tier1/dynamic/rare

    # dynamic / generation bottleneck
    "136b0064",  # UNSOLVED, hard_tier1/dynamic/rare
    "269e22fb",  # UNSOLVED, dynamic + multi-query
    "0934a4d8",  # UNSOLVED, tier3/dynamic/high_ratio/extreme_ratio
    "20a9e565",  # UNSOLVED, tier3/dynamic/multi-query/high_ratio/extreme_ratio
    "89565ca0",  # UNSOLVED, tier3/dynamic/high_ratio/extreme_ratio

    # solved but dual missed -> selection-depth test
    "4c7dc4dd",  # SOLVED, Dual No in 48-run, rank 4; tier3/multi-query/extreme_ratio
    "7b0280bc",  # SOLVED, sometimes Dual No; rank 1-2 depending run
    "2d0172a1",  # SOLVED, Dual No, rank 2/2

    # extra solved but dual missed -> selection-depth test
    "31f7f899",  # SOLVED, Dual No, rank 2/2
    "3dc255db",  # SOLVED, Dual No, rank 3/3
    "8f3a5a89",  # SOLVED, Dual No, rank 2/2
    "800d221b",  # SOLVED, Dual No, rank 2/2

    # solved + dual yes -> guardrail
    "6e453dd6",  # SOLVED, Dual Yes
    "7b80bb43",  # SOLVED, Dual Yes
    "1818057f",  # SOLVED, Dual Yes
    "4a21e3da",  # SOLVED, Dual Yes, multi-query

    # unsolved but useful buckets
    "cb2d8a2c",  # UNSOLVED, easy_static + multi-query
    "58490d8a",  # UNSOLVED, dynamic + high_ratio-ish
]

DEEP_POOL_KEYS = {
    "4c7dc4dd",
    "7b0280bc",
    "2d0172a1",
    "31f7f899",
    "3dc255db",
    "8f3a5a89",
    "800d221b",
}

def puzzle_base_keys(puzzle_id):
    return sorted([k for k in decoder.decoded_results.keys() if k.startswith(puzzle_id + "_")])

def is_correct(arr, label):
    return np.array_equal(np.asarray(arr), np.asarray(label))

def rank_of_correct(ranked, label):
    for i, cand in enumerate(ranked):
        if is_correct(cand, label):
            return i + 1
    return None

def make_dual_selection(pool_k_default=3, pool_k_deep=8, deep_keys=None):
    if deep_keys is None:
        deep_keys = set()

    selected = {}
    meta = {}

    for bk, guesses in decoder.decoded_results.items():
        g = {k: v for k, v in guesses.items()}
        ranked_a = score_full_probmul_3(g)
        ranked_b = score_kgmon(g)

        base_key = bk.split("_")[0]
        pool_k = pool_k_deep if base_key in deep_keys else pool_k_default

        candidates_pool = []
        seen = set()
        for cand in ranked_a[:pool_k] + ranked_b[:pool_k]:
            h = hashable(cand)
            if h not in seen:
                seen.add(h)
                candidates_pool.append(cand)

        top_a = candidates_pool[0] if candidates_pool else None
        top_b = candidates_pool[1] if len(candidates_pool) > 1 else top_a
        selected[bk] = [x for x in [top_a, top_b] if x is not None]

        meta[bk] = {
            "pool_k": pool_k,
            "pool_size": len(candidates_pool),
            "ranked_a_len": len(ranked_a),
            "ranked_b_len": len(ranked_b),
        }

    return selected, meta

def score_selected(selected):
    correct = set()
    num_tasks_per_puzzle = {}

    for bk in decoder.decoded_results:
        puzzle_id, sub = bk.rsplit("_", 1)
        num_tasks_per_puzzle[puzzle_id] = max(num_tasks_per_puzzle.get(puzzle_id, 0), int(sub) + 1)

    for bk, guesses in selected.items():
        if bk not in decoder.dataset.replies:
            continue
        label = decoder.dataset.replies[bk][0]
        if any(is_correct(g, label) for g in guesses[:decoder.n_guesses]):
            correct.add(bk)

    score = sum(1 / num_tasks_per_puzzle[bk.rsplit("_", 1)[0]] for bk in correct)
    return correct, score

top3_selected, top3_meta = make_dual_selection(pool_k_default=3, pool_k_deep=3, deep_keys=set())
cond8_selected, cond8_meta = make_dual_selection(pool_k_default=3, pool_k_deep=8, deep_keys=DEEP_POOL_KEYS)
actual_selected = decoder.run_dual_selection()

top3_correct, top3_score = score_selected(top3_selected)
cond8_correct, cond8_score = score_selected(cond8_selected)
actual_correct, actual_score = score_selected(actual_selected)

summary = []

for puzzle_id in DEBUG_PUZZLES:
    bks = puzzle_base_keys(puzzle_id)

    if not bks:
        summary.append({
            "id": puzzle_id,
            "status": "NO_OUTPUT",
            "queries": 0,
            "candidate_solved": False,
            "actual_dual": False,
            "top3": False,
            "cond8": False,
            "best_rank_probmul": "N/A",
            "best_rank_kgmon": "N/A",
            "best_beam": "N/A",
        })
        continue

    candidate_solved = False
    actual_dual = False
    top3_hit = False
    cond8_hit = False
    best_rank_a = None
    best_rank_b = None
    best_beam = None

    for bk in bks:
        if bk not in decoder.dataset.replies:
            continue

        label = decoder.dataset.replies[bk][0]
        guesses = decoder.decoded_results.get(bk, {})
        ranked_a = score_full_probmul_3(guesses)
        ranked_b = score_kgmon(guesses)

        ra = rank_of_correct(ranked_a, label)
        rb = rank_of_correct(ranked_b, label)

        if ra is not None or rb is not None:
            candidate_solved = True
            if ra is not None:
                best_rank_a = ra if best_rank_a is None else min(best_rank_a, ra)
            if rb is not None:
                best_rank_b = rb if best_rank_b is None else min(best_rank_b, rb)

        for sample in guesses.values():
            if is_correct(sample["solution"], label):
                beam = sample.get("beam_score")
                if beam is not None and (best_beam is None or beam < best_beam):
                    best_beam = beam

        if any(is_correct(g, label) for g in actual_selected.get(bk, [])[:decoder.n_guesses]):
            actual_dual = True
        if bk in top3_correct:
            top3_hit = True
        if bk in cond8_correct:
            cond8_hit = True

    summary.append({
        "id": puzzle_id,
        "status": "SOLVED" if candidate_solved else "UNSOLVED",
        "queries": len(bks),
        "candidate_solved": candidate_solved,
        "actual_dual": actual_dual,
        "top3": top3_hit,
        "cond8": cond8_hit,
        "best_rank_probmul": best_rank_a if best_rank_a is not None else "N/A",
        "best_rank_kgmon": best_rank_b if best_rank_b is not None else "N/A",
        "best_beam": f"{best_beam:.5f}" if best_beam is not None else "N/A",
    })

print("=" * 100)
print("CHECK 1: SOLVE / SELECTION SUMMARY")
print("=" * 100)
print(
    f"{'Puzzle':<12} {'Status':<10} {'Q':>2} "
    f"{'Top3':<6} {'Cond8':<6} {'Actual':<7} "
    f"{'RankA':<6} {'RankB':<6} {'BestBeam':<10}"
)
print("-" * 100)

for r in summary:
    print(
        f"{r['id']:<12} {r['status']:<10} {r['queries']:>2} "
        f"{str(r['top3']):<6} {str(r['cond8']):<6} {str(r['actual_dual']):<7} "
        f"{str(r['best_rank_probmul']):<6} {str(r['best_rank_kgmon']):<6} {r['best_beam']:<10}"
    )

no_output = [r["id"] for r in summary if r["status"] == "NO_OUTPUT"]
unsolved = [r["id"] for r in summary if r["status"] == "UNSOLVED"]
candidate_solved = [r["id"] for r in summary if r["candidate_solved"]]
actual_dual_solved = [r["id"] for r in summary if r["actual_dual"]]
cond8_solved = [r["id"] for r in summary if r["cond8"]]

print("\nTotals:")
print(f"  no_output:        {len(no_output)} -> {no_output}")
print(f"  unsolved:         {len(unsolved)} -> {unsolved}")
print(f"  candidate_solved: {len(candidate_solved)} -> {candidate_solved}")
print(f"  actual_dual:      {len(actual_dual_solved)} -> {actual_dual_solved}")
print(f"  cond8_dual:       {len(cond8_solved)} -> {cond8_solved}")

print("\nScores on decoded subkeys:")
print(f"  top3_score:   {top3_score:.3f} correct={sorted(top3_correct)}")
print(f"  cond8_score:  {cond8_score:.3f} correct={sorted(cond8_correct)}")
print(f"  actual_score: {actual_score:.3f} correct={sorted(actual_correct)}")

rescued = sorted(cond8_correct - top3_correct)
lost = sorted(top3_correct - cond8_correct)
print(f"\nRescued by conditional top-8: {rescued}")
print(f"Lost by conditional top-8:    {lost}")

print("\nDeep-pool details:")
for bk in sorted(decoder.decoded_results):
    base = bk.split("_")[0]
    if base not in DEEP_POOL_KEYS:
        continue
    print(
        f"  {bk:<12} "
        f"top3={bk in top3_correct} "
        f"cond8={bk in cond8_correct} "
        f"actual={bk in actual_correct} "
        f"pool_size={cond8_meta.get(bk, {}).get('pool_size')}"
    )

print("\n" + "=" * 100)
print("CHECK 2: MULTI-QUERY BREAKDOWN")
print("=" * 100)

for r in summary:
    if r["queries"] > 1:
        print(
            f"{r['id']:<12} queries={r['queries']} "
            f"status={r['status']} top3={r['top3']} cond8={r['cond8']} actual={r['actual_dual']}"
        )

print("\n" + "=" * 100)
print("CHECK 3: TIER SUMMARY")
print("=" * 100)

tier_stats = {}
for r in summary:
    if r["id"] not in data_dict:
        continue
    tier, fine = get_puzzle_priority(r["id"], data_dict)
    s = tier_stats.setdefault(tier, {"total": 0, "no_output": 0, "candidate": 0, "actual": 0, "cond8": 0})
    s["total"] += 1
    s["no_output"] += int(r["status"] == "NO_OUTPUT")
    s["candidate"] += int(r["candidate_solved"])
    s["actual"] += int(r["actual_dual"])
    s["cond8"] += int(r["cond8"])

for tier in sorted(tier_stats):
    s = tier_stats[tier]
    print(
        f"Tier {tier}: total={s['total']} no_output={s['no_output']} "
        f"candidate={s['candidate']} actual={s['actual']} cond8={s['cond8']}"
    )

print("=" * 100)
print("DIAGNOSTIC COMPLETE.")
print("=" * 100)